# B3: DINOv2 + ConvNeXt Ensemble — 5-Fold Cross-Validation.

**Author:** Mridankan Mandal.

**Competition:** [CSIRO Image2Biomass](https://www.kaggle.com/competitions/csiro-biomass).

---

## Overview.

This notebook evaluates a **zero-shot ensemble** (DINOv2-Base + ConvNeXt-Base) using
5-fold Stratified Group K-Fold cross-validation on the **training set**.
Models are pretrained only (no fine-tuning on biomass data).

### Architecture.
- **DINOv2-Base:** `vit_base_patch14_dinov2.lvd142m` via `timm`.
- **ConvNeXt-Base:** `convnext_base.fb_in22k_ft_in1k_384` via `timm`.
- **Ensemble:** Weighted average (0.55 DINOv2, 0.45 ConvNeXt).
- **Cross-Validation:** 5-fold Stratified Group K-Fold (same splits as B4/proposed model).

### Requirements.
- Kaggle Internet must be enabled for Hugging Face downloads.
- No GPU required but recommended for faster inference.


In [ ]:
# Install helper libraries for Hugging Face model downloads.
!pip install -q huggingface_hub safetensors


In [ ]:
import os
import gc
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import timm
from timm.data import resolve_model_data_config
from torchvision import transforms
from torch.amp import autocast
from sklearn.model_selection import StratifiedGroupKFold

# --- Configuration ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = '/kaggle/input/competitions/csiro-biomass'
TRAIN_CSV_PATH = os.path.join(DATA_DIR, 'train.csv')
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train')

SEED = 17
N_FOLDS = 5
NUM_TARGETS = 5
TARGET_NAMES = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
COMP_WEIGHTS = np.array([0.1, 0.1, 0.1, 0.2, 0.5])

# Model identifiers (timm will auto-download from HuggingFace).
DINO_MODEL_NAME = 'vit_base_patch14_dinov2.lvd142m'
CONV_MODEL_NAME = 'convnext_base.fb_in22k_ft_in1k_384'

# Ensemble weights.
W_DINO = 0.55
W_CONV = 0.45

np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'Device: {DEVICE}')
print(f'DINOv2 model: {DINO_MODEL_NAME}')
print(f'ConvNeXt model: {CONV_MODEL_NAME}')


In [ ]:
# --- Load & pivot training data (same as B4/proposed model) ---
df_long = pd.read_csv(TRAIN_CSV_PATH)
df_long['image_id'] = df_long['sample_id'].str.split('__').str[0]

df_wide = df_long.pivot_table(
    index=['image_id', 'image_path'],
    columns='target_name',
    values='target',
    aggfunc='first'
).reset_index()

for col in TARGET_NAMES:
    if col not in df_wide.columns:
        df_wide[col] = 0.0

# --- Create fold splits (same as B4/proposed model) ---
df_wide['total_bin'] = pd.qcut(df_wide['Dry_Total_g'], q=5, labels=False, duplicates='drop')

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
df_wide['fold'] = -1
for fold, (_, val_idx) in enumerate(sgkf.split(df_wide, df_wide['total_bin'], groups=df_wide['image_id'])):
    df_wide.loc[val_idx, 'fold'] = fold

print(f'Training images: {len(df_wide)}')
print(f'Fold distribution:\n{df_wide["fold"].value_counts().sort_index()}')


In [ ]:
def weighted_r2_score(y_true, y_pred):
    """Compute weighted R² score (same as B4/proposed model)."""
    r2_scores = []
    for i in range(y_true.shape[1]):
        yt, yp = y_true[:, i], y_pred[:, i]
        ss_res = np.sum((yt - yp) ** 2)
        ss_tot = np.sum((yt - np.mean(yt)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        r2_scores.append(r2)
    r2_scores = np.array(r2_scores)
    weighted = np.sum(r2_scores * COMP_WEIGHTS) / np.sum(COMP_WEIGHTS)
    return weighted, r2_scores


def run_model_inference(model_name, model_label, img_ids, img_dir):
    """Run zero-shot inference for a single timm model on given image IDs."""
    model = timm.create_model(model_name, pretrained=True, num_classes=NUM_TARGETS)
    model = model.to(DEVICE)
    model.eval()

    # Resolve the model's native input size.
    data_cfg = resolve_model_data_config(model)
    img_size = data_cfg['input_size'][-1]

    val_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    preds = np.zeros((len(img_ids), NUM_TARGETS), dtype=np.float32)

    with torch.no_grad():
        for idx, img_id in enumerate(img_ids):
            img_path = os.path.join(img_dir, f'{img_id}.jpg')
            if not os.path.exists(img_path):
                continue
            img = Image.open(img_path).convert('RGB')
            img_t = val_tfm(img).unsqueeze(0).to(DEVICE)
            with autocast(device_type=DEVICE):
                out = model(img_t)
            preds[idx] = out.cpu().numpy()[0]

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return preds


# --- Run zero-shot inference on ALL training images (once per model) ---
print('Note: Using pretrained base models (not fine-tuned on biomass data).')
print('Running inference on all training images...\n')

all_img_ids = df_wide['image_id'].values

print(f'Running DINOv2 ({DINO_MODEL_NAME})...')
preds_dino_all = run_model_inference(DINO_MODEL_NAME, 'DINOv2', all_img_ids, TRAIN_IMG_DIR)
print(f'  DINOv2 done: {preds_dino_all.shape}')

print(f'Running ConvNeXt ({CONV_MODEL_NAME})...')
preds_conv_all = run_model_inference(CONV_MODEL_NAME, 'ConvNeXt', all_img_ids, TRAIN_IMG_DIR)
print(f'  ConvNeXt done: {preds_conv_all.shape}')

# Ensemble predictions.
preds_ensemble_all = W_DINO * preds_dino_all + W_CONV * preds_conv_all
print(f'\nEnsemble predictions: {preds_ensemble_all.shape}')


In [ ]:
# --- Evaluate per-fold using the precomputed ensemble predictions ---
y_true_all = df_wide[TARGET_NAMES].values
fold_results = []

for fold in range(N_FOLDS):
    val_mask = (df_wide['fold'] == fold).values
    y_true_fold = y_true_all[val_mask]
    y_pred_fold = preds_ensemble_all[val_mask]

    w_r2, per_target = weighted_r2_score(y_true_fold, y_pred_fold)
    fold_results.append({'fold': fold, 'weighted_r2': w_r2, 'per_target_r2': per_target})

    print(f'Fold {fold}: Weighted R² = {w_r2:.6f}')
    for j, col in enumerate(TARGET_NAMES):
        print(f'  {col}: R² = {per_target[j]:.6f}')

# --- Summary ---
w_r2_all = [r['weighted_r2'] for r in fold_results]
per_target_arr = np.array([r['per_target_r2'] for r in fold_results])

print(f'\n{"="*60}')
print(f'B3: DINOv2 + ConvNeXt Zero-Shot Ensemble — {N_FOLDS}-Fold CV Results')
print(f'{"="*60}')
print(f'Mean Weighted R²: {np.mean(w_r2_all):.6f} ± {np.std(w_r2_all):.6f}')
print()
for j, col in enumerate(TARGET_NAMES):
    print(f'  {col}: {np.mean(per_target_arr[:, j]):.6f} ± {np.std(per_target_arr[:, j]):.6f}')

# Also report individual model performance.
print(f'\n--- Individual Model Performance (overall, not per-fold) ---')
w_r2_dino, per_dino = weighted_r2_score(y_true_all, preds_dino_all)
w_r2_conv, per_conv = weighted_r2_score(y_true_all, preds_conv_all)
print(f'DINOv2 only:  Weighted R² = {w_r2_dino:.6f}')
print(f'ConvNeXt only: Weighted R² = {w_r2_conv:.6f}')
print(f'Ensemble:      Weighted R² = {np.mean(w_r2_all):.6f}')
